### This notebook conducts an ablation study to evaluate `URL` vs `HTML` vs `JS` features on both Datasets

| feature set | columns |
|---|---|
| `url_only`, `html_only`, `js_only` | one group |
| `no_url`, `no_html`, `no_js` | everything except one group |
| `full` | all features |

In [12]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix, roc_curve)

FEATURES_PATH = "DA_features/features.csv"
MODELS_DIR = "models_final_E1"
RESULTS_DIR = "results_final_E1"
SEED = 42

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [13]:
df = pd.read_csv(FEATURES_PATH)

# feature groups
url_cols = [c for c in df.columns if c.startswith("url_")]
html_cols = [c for c in df.columns if c.startswith("html_")]
js_cols = [c for c in df.columns if c.startswith("js_") or c.startswith("tfidf_bpe_")]
feature_cols = url_cols + html_cols + js_cols
print("url:", len(url_cols), "| html:", len(html_cols), "| js:", len(js_cols), "| total:", len(feature_cols))

df[feature_cols] = df[feature_cols].fillna(0)

# use the train/test split that is already in the csv
train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
y_train = train_df["label"].values
y_test = test_df["label"].values
print("train:", len(train_df), "| test:", len(test_df))

url: 12 | html: 14 | js: 103 | total: 129
train: 13815 | test: 3454


In [14]:
with open(f"{RESULTS_DIR}/best_params_E1.json") as f:
    best = json.load(f)

feature_sets = {
    "url_only": url_cols,
    "html_only": html_cols,
    "js_only": js_cols,
    "no_url": html_cols + js_cols,
    "no_html": url_cols + js_cols,
    "no_js": url_cols + html_cols,
}
{name: len(cols) for name, cols in feature_sets.items()}

{'url_only': 12,
 'html_only': 14,
 'js_only': 103,
 'no_url': 117,
 'no_html': 115,
 'no_js': 26}

In [ ]:
results = []

for set_name, cols in feature_sets.items():
    # scaler fitted on the training split of this feature set only
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(train_df[cols])
    X_test = scaler.transform(test_df[cols])

    models = {
        "decision_tree": DecisionTreeClassifier(random_state=SEED, **best["decision_tree"]),
        "random_forest": RandomForestClassifier(random_state=SEED, n_jobs=-1, **best["random_forest"]),
        "xgboost": XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss", **best["xgboost"]),
        "stacking": StackingClassifier(
            estimators=[("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **best["random_forest"])),
                        ("xgb", XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss", **best["xgboost"]))],
            final_estimator=LogisticRegression(solver="saga", max_iter=5000, random_state=SEED,
                                               C=best["stacking"]["final_estimator__C"],
                                               l1_ratio=best["stacking"]["final_estimator__l1_ratio"]),
            cv=5),
    }

    for name, model in models.items():
        # training time
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        train_time = time.perf_counter() - t0

        # inference time on the whole test set (median of 5 runs, after a warm-up call)
        model.predict(X_test[:64])
        runs = []
        for _ in range(5):
            t0 = time.perf_counter()
            y_pred = model.predict(X_test)
            runs.append(time.perf_counter() - t0)
        infer_batch = float(np.median(runs))

        # inference time for a single sample
        single = []
        for i in range(min(200, len(X_test))):
            t0 = time.perf_counter()
            model.predict(X_test[i:i + 1])
            single.append(time.perf_counter() - t0)

        y_score = model.predict_proba(X_test)[:, 1]
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        results.append({
            "feature_set": set_name,
            "model": name,
            "n_features": len(cols),
                "n_train": len(y_train),
                "n_test": len(y_test),
                "train_time_s": train_time,
                "infer_test_set_s": infer_batch,
                "infer_ms_per_sample": 1000 * infer_batch / len(X_test),
                "infer_single_row_ms": 1000 * float(np.median(single)),
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "macro_f1": f1_score(y_test, y_pred, average="macro"),
                "mcc": matthews_corrcoef(y_test, y_pred),
                "roc_auc": roc_auc_score(y_test, y_score),
                "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        })
        print(set_name, " | ", name, "| train", round(train_time, 4), "s | Macro F1", round(f1_score(y_test, y_pred, average="macro"), 4))

ablation_df = pd.DataFrame(results)
ablation_df.to_csv(f"{RESULTS_DIR}/ablation_results_E1.csv", index=False)
ablation_df

url_only  |  decision_tree | train 0.0128 s | Macro F1 0.9935
url_only  |  random_forest | train 0.1843 s | Macro F1 0.9919
url_only  |  xgboost | train 0.074 s | Macro F1 0.9938
url_only  |  stacking | train 1.5432 s | Macro F1 0.9941
html_only  |  decision_tree | train 0.0264 s | Macro F1 0.8278
html_only  |  random_forest | train 0.3931 s | Macro F1 0.8748
html_only  |  xgboost | train 0.0761 s | Macro F1 0.8618
html_only  |  stacking | train 2.5048 s | Macro F1 0.8775
js_only  |  decision_tree | train 0.1285 s | Macro F1 0.7712
js_only  |  random_forest | train 0.7177 s | Macro F1 0.8563
js_only  |  xgboost | train 0.1346 s | Macro F1 0.8434
js_only  |  stacking | train 4.2822 s | Macro F1 0.8546
no_url  |  decision_tree | train 0.1605 s | Macro F1 0.8255
no_url  |  random_forest | train 0.8114 s | Macro F1 0.8932
no_url  |  xgboost | train 0.1442 s | Macro F1 0.8743
no_url  |  stacking | train 4.9136 s | Macro F1 0.8929
no_html  |  decision_tree | train 0.1268 s | Macro F1 0.9925


,feature_set,model,n_features,n_train,n_test,train_time_s,infer_test_set_s,infer_ms_per_sample,infer_single_row_ms,accuracy,precision,recall,macro_f1,mcc,roc_auc,tn,fp,fn,tp
0,url_only,decision_tree,12,13815,3454,0.012790,0.000078,0.000022,0.024604,0.994210,0.999123,0.983592,0.993479,0.987027,0.995326,2295,1,19,1139
1,url_only,random_forest,12,13815,3454,0.184267,0.013594,0.003936,13.447813,0.992762,0.992181,0.986183,0.991868,0.983747,0.997271,2287,9,16,1142
2,url_only,xgboost,12,13815,3454,0.074023,0.000447,0.000129,0.149729,0.994499,0.999124,0.984456,0.993807,0.987674,0.997886,2295,1,18,1140
3,url_only,stacking,12,13815,3454,1.543184,0.014202,0.004112,13.632771,0.994789,0.997382,0.987047,0.994139,0.988308,0.997898,2293,3,15,1143
4,html_only,decision_tree,14,13815,3454,0.026412,0.000084,0.000024,0.023125,0.849450,0.797575,0.738342,0.827829,0.656966,0.895524,2079,217,303,855
5,html_only,random_forest,14,13815,3454,0.393061,0.024824,0.007187,13.334146,0.889983,0.856227,0.807427,0.874766,0.750345,0.944179,2139,157,223,935
6,html_only,xgboost,14,13815,3454,0.076133,0.000434,0.000126,0.152416,0.876665,0.814974,0.817789,0.861764,0.723531,0.937087,2081,215,211,947
7,html_only,stacking,14,13815,3454,2.504834,0.025671,0.007432,13.666209,0.892588,0.862673,0.808290,0.877534,0.756070,0.947396,2147,149,222,936
8,js_only,decision_tree,103,13815,3454,0.128521,0.000147,0.000043,0.025125,0.799653,0.716141,0.666667,0.771201,0.543443,0.861746,1990,306,386,772
9,js_only,random_forest,103,13815,3454,0.717666,0.026475,0.007665,13.288250,0.871453,0.803571,0.816062,0.856349,0.712755,0.929507,2065,231,213,945


In [17]:
order = ["url_only", "html_only", "js_only", "no_url", "no_html", "no_js"]
model_order = ["decision_tree", "random_forest", "xgboost", "stacking"]
for metric in ["macro_f1", "mcc", "accuracy"]:
    print(metric)
    display(ablation_df.pivot(index="feature_set", columns="model", values=metric).loc[order, model_order].round(4))

macro_f1


model,decision_tree,random_forest,xgboost,stacking
feature_set,,,,
url_only,0.9935,0.9919,0.9938,0.9941
html_only,0.8278,0.8748,0.8618,0.8775
js_only,0.7712,0.8563,0.8434,0.8546
no_url,0.8255,0.8932,0.8743,0.8929
no_html,0.9925,0.9948,0.9958,0.9958
no_js,0.9922,0.9951,0.9961,0.9958


mcc


model,decision_tree,random_forest,xgboost,stacking
feature_set,,,,
url_only,0.9870,0.9837,0.9877,0.9883
html_only,0.6570,0.7503,0.7235,0.7561
js_only,0.5434,0.7128,0.6871,0.7093
no_url,0.6511,0.7881,0.7485,0.7870
no_html,0.9851,0.9896,0.9916,0.9916
no_js,0.9844,0.9903,0.9922,0.9916


accuracy


model,decision_tree,random_forest,xgboost,stacking
feature_set,,,,
url_only,0.9942,0.9928,0.9945,0.9948
html_only,0.8494,0.8900,0.8767,0.8926
js_only,0.7997,0.8715,0.8593,0.8700
no_url,0.8451,0.9068,0.8877,0.9062
no_html,0.9933,0.9954,0.9962,0.9962
no_js,0.9931,0.9957,0.9965,0.9962
